In [1]:
FRAMEWORK = 'polars'

# Proyecto Big Data

## 0. Instalación, entorno y acceso a los datos

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q polars pyarrow pandas gcsfs
print('Entorno:', 'Google Colab' if IN_COLAB else 'local')

Entorno: local


In [3]:
from pathlib import Path

# El dataset se lee directamente desde el bucket de Google Cloud Storage,
# que es el data lake del proyecto. De esta forma el procesamiento consume
# los datos del bucket y no una copia local.
BUCKET = 'gs://bank-segmentation-bigdata-data'
DATA_PATH = f'{BUCKET}/raw/bank_transactions.csv'

if IN_COLAB:
    # Autenticación necesaria para que Colab pueda leer del bucket.
    from google.colab import auth
    auth.authenticate_user()

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / FRAMEWORK
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset: {DATA_PATH}')
print(f'Resultados: {OUTPUT_DIR}')

Dataset: gs://bank-segmentation-bigdata-data/raw/bank_transactions.csv
Resultados: /home/neo/nb2/entrega_polars_dask/outputs/polars


## 1. Carga e inspección con Polars (modo lazy)

In [4]:
import math
import polars as pl

AMOUNT = 'TransactionAmount (INR)'
BALANCE = 'CustAccountBalance'

# Esquema explícito
SCHEMA = {
    'TransactionID': pl.String,
    'CustomerID': pl.String,
    'CustomerDOB': pl.String,
    'CustGender': pl.String,
    'CustLocation': pl.String,
    BALANCE: pl.Float64,
    'TransactionDate': pl.String,
    'TransactionTime': pl.Int64,
    AMOUNT: pl.Float64,
}

# scan_csv: devuelve un LazyFrame con el plan de consulta.
raw = pl.scan_csv(DATA_PATH, schema_overrides=SCHEMA, null_values=['', 'nan'])

n_raw = raw.select(pl.len()).collect().item()
print('Versión Polars:', pl.__version__)
print('Hilos disponibles:', pl.thread_pool_size())
print('Filas:', n_raw, '| Columnas:', len(raw.collect_schema()))
print(raw.collect_schema())
display(raw.head().collect())

Versión Polars: 1.44.2
Hilos disponibles: 4
Filas: 1048567 | Columnas: 9
Schema([('TransactionID', String), ('CustomerID', String), ('CustomerDOB', String), ('CustGender', String), ('CustLocation', String), ('CustAccountBalance', Float64), ('TransactionDate', String), ('TransactionTime', Int64), ('TransactionAmount (INR)', Float64)])


TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR)
str,str,str,str,str,f64,str,i64,f64
"""T1""","""C5841053""","""10/1/94""","""F""","""JAMSHEDPUR""",17819.05,"""2/8/16""",143207,25.0
"""T2""","""C2142763""","""4/4/57""","""M""","""JHAJJAR""",2270.69,"""2/8/16""",141858,27999.0
"""T3""","""C4417068""","""26/11/96""","""F""","""MUMBAI""",17874.44,"""2/8/16""",142712,459.0
"""T4""","""C5342380""","""14/9/73""","""F""","""MUMBAI""",866503.21,"""2/8/16""",142714,2060.0
"""T5""","""C9031234""","""24/3/88""","""F""","""NAVI MUMBAI""",6714.43,"""2/8/16""",181156,1762.5


## 2. Preparación

In [5]:
transaction_date = pl.col('TransactionDate').str.to_date('%d/%m/%y', strict=False)
txn_year = transaction_date.dt.year()

dob_parts = pl.col('CustomerDOB').str.split('/')
dob_day = dob_parts.list.get(0, null_on_oob=True).cast(pl.Int32, strict=False)
dob_month = dob_parts.list.get(1, null_on_oob=True).cast(pl.Int32, strict=False)
dob_year_raw = dob_parts.list.get(2, null_on_oob=True).cast(pl.Int32, strict=False)

birth_year = (
    pl.when(dob_year_raw < 100)
    .then(
        1900 + dob_year_raw
        + pl.when(dob_year_raw <= (txn_year % 100)).then(100).otherwise(0)
    )
    .otherwise(dob_year_raw)
)

birthday_not_reached = (
    pl.when(
        (transaction_date.dt.month() < dob_month)
        | (
            (transaction_date.dt.month() == dob_month)
            & (transaction_date.dt.day() < dob_day)
        )
    ).then(1).otherwise(0)
)

age_raw = txn_year - birth_year - birthday_not_reached

base = raw.with_columns(
    TransactionDateParsed=transaction_date,
    LocationNormalized=pl.col('CustLocation').str.strip_chars().str.to_uppercase(),
    Age=pl.when(age_raw.is_between(18, 100)).then(age_raw),
    Hour=(pl.col('TransactionTime') // 10000).cast(pl.Int32),
).with_columns(
    TimeBand=pl.when(pl.col('Hour').is_between(0, 5)).then(pl.lit('Madrugada'))
    .when(pl.col('Hour').is_between(6, 11)).then(pl.lit('Manana'))
    .when(pl.col('Hour').is_between(12, 17)).then(pl.lit('Tarde'))
    .when(pl.col('Hour').is_between(18, 23)).then(pl.lit('Noche'))
    .otherwise(pl.lit('Invalida')),
    AgeRange=pl.when(pl.col('Age').is_between(18, 25)).then(pl.lit('18-25'))
    .when(pl.col('Age').is_between(26, 35)).then(pl.lit('26-35'))
    .when(pl.col('Age').is_between(36, 45)).then(pl.lit('36-45'))
    .when(pl.col('Age').is_between(46, 60)).then(pl.lit('46-60'))
    .when(pl.col('Age').is_between(61, 100)).then(pl.lit('61-100'))
    .otherwise(pl.lit('Desconocido')),
)

clean_lazy = (
    base
    .unique(subset=['TransactionID'], keep='first', maintain_order=True)
    .unique(
        subset=['CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT],
        keep='first', maintain_order=True,
    )
)

clean = clean_lazy.collect()
print('Filas preparadas:', clean.height)
display(clean.head())

Filas preparadas: 1048567


TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR),TransactionDateParsed,LocationNormalized,Age,Hour,TimeBand,AgeRange
str,str,str,str,str,f64,str,i64,f64,date,str,i32,i32,str,str
"""T1""","""C5841053""","""10/1/94""","""F""","""JAMSHEDPUR""",17819.05,"""2/8/16""",143207,25.0,2016-08-02,"""JAMSHEDPUR""",22,14,"""Tarde""","""18-25"""
"""T2""","""C2142763""","""4/4/57""","""M""","""JHAJJAR""",2270.69,"""2/8/16""",141858,27999.0,2016-08-02,"""JHAJJAR""",59,14,"""Tarde""","""46-60"""
"""T3""","""C4417068""","""26/11/96""","""F""","""MUMBAI""",17874.44,"""2/8/16""",142712,459.0,2016-08-02,"""MUMBAI""",19,14,"""Tarde""","""18-25"""
"""T4""","""C5342380""","""14/9/73""","""F""","""MUMBAI""",866503.21,"""2/8/16""",142714,2060.0,2016-08-02,"""MUMBAI""",42,14,"""Tarde""","""36-45"""
"""T5""","""C9031234""","""24/3/88""","""F""","""NAVI MUMBAI""",6714.43,"""2/8/16""",181156,1762.5,2016-08-02,"""NAVI MUMBAI""",28,18,"""Noche""","""26-35"""


### Plan de consulta

In [6]:
print(clean_lazy.explain())

UNIQUE[maintain_order: true, keep_strategy: First] BY Some(["CustomerID", "TransactionDate", "TransactionTime", "TransactionAmount (INR)"])
  UNIQUE[maintain_order: true, keep_strategy: First] BY Some(["TransactionID"])
     WITH_COLUMNS:
     [when(col("Hour").is_between([0, 5])).then("Madrugada").otherwise(when(col("Hour").is_between([6, 11])).then("Manana").otherwise(when(col("Hour").is_between([12, 17])).then("Tarde").otherwise(when(col("Hour").is_between([18, 23])).then("Noche").otherwise("Invalida")))).alias("TimeBand"), when(col("Age").is_between([18, 25])).then("18-25").otherwise(when(col("Age").is_between([26, 35])).then("26-35").otherwise(when(col("Age").is_between([36, 45])).then("36-45").otherwise(when(col("Age").is_between([46, 60])).then("46-60").otherwise(when(col("Age").is_between([61, 100])).then("61-100").otherwise("Desconocido"))))).alias("AgeRange")] 
      simple π 13/13 ["TransactionID", "CustomerID", ... 11 other columns]
         WITH_COLUMNS:
         [col("__P

## Pregunta 1. Diagnóstico de calidad de datos

In [7]:
%%time
null_counts = raw.select(pl.all().null_count()).collect()
q01 = pl.DataFrame({
    'column': null_counts.columns,
    'null_count': [null_counts[c].item() for c in null_counts.columns],
}).with_columns(
    null_percent=(pl.col('null_count') * 100 / n_raw).round(4)
)
q01.write_csv(OUTPUT_DIR / 'q01_calidad_datos.csv')
display(q01)

column,null_count,null_percent
str,i64,f64
"""TransactionID""",0,0.0
"""CustomerID""",0,0.0
"""CustomerDOB""",3397,0.324
"""CustGender""",1100,0.1049
"""CustLocation""",151,0.0144
"""CustAccountBalance""",2369,0.2259
"""TransactionDate""",0,0.0
"""TransactionTime""",0,0.0
"""TransactionAmount (INR)""",0,0.0


CPU times: user 1.01 s, sys: 396 ms, total: 1.41 s
Wall time: 1.58 s


## Pregunta 2. Detección y tratamiento de duplicados

In [8]:
%%time
def duplicate_excess(columns):
    return (
        raw.group_by(columns).len()
        .filter(pl.col('len') > 1)
        .select((pl.col('len') - 1).sum().fill_null(0))
        .collect().item()
    )

q02 = pl.DataFrame({
    'criterion': [
        'TransactionID',
        'CustomerID+TransactionDate+Amount',
        'CustomerID+TransactionDate+Time+Amount',
    ],
    'duplicate_rows': [
        duplicate_excess(['TransactionID']),
        duplicate_excess(['CustomerID', 'TransactionDate', AMOUNT]),
        duplicate_excess(['CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT]),
    ],
    'treatment': [
        'Eliminar repetidos',
        'Conservar y revisar: ocurren a horas distintas',
        'Eliminar repetidos exactos del evento',
    ],
})
q02.write_csv(OUTPUT_DIR / 'q02_duplicados.csv')
display(q02)

criterion,duplicate_rows,treatment
str,i64,str
"""TransactionID""",0,"""Eliminar repetidos"""
"""CustomerID+TransactionDate+Amo…",31,"""Conservar y revisar: ocurren a…"
"""CustomerID+TransactionDate+Tim…",0,"""Eliminar repetidos exactos del…"


CPU times: user 3.25 s, sys: 1.02 s, total: 4.27 s
Wall time: 2.24 s


## Pregunta 3. Edad exacta y rangos etarios

In [9]:
%%time
q03 = clean.group_by('AgeRange').agg(
    transactions=pl.len(),
    mean_age=pl.col('Age').mean().round(2),
).sort('AgeRange')
q03.write_csv(OUTPUT_DIR / 'q03_edades.csv')
display(q03)

AgeRange,transactions,mean_age
str,u32,f64
"""18-25""",295117,23.08
"""26-35""",482990,29.62
"""36-45""",142585,39.43
"""46-60""",50856,51.03
"""61-100""",14303,67.1
"""Desconocido""",62716,null


CPU times: user 73.2 ms, sys: 2.42 ms, total: 75.6 ms
Wall time: 37.4 ms


## Pregunta 4. Franja horaria y ubicación normalizada

In [10]:
%%time
q04 = clean.group_by('TimeBand').agg(
    transactions=pl.len(),
    unique_locations=pl.col('LocationNormalized').drop_nulls().n_unique(),
).sort(['transactions', 'TimeBand'], descending=[True, False])
q04.write_csv(OUTPUT_DIR / 'q04_franja_ubicacion.csv')
display(q04)

TimeBand,transactions,unique_locations
str,u32,u32
"""Noche""",436179,7246
"""Tarde""",390884,7146
"""Manana""",174473,4889
"""Madrugada""",47031,2208


CPU times: user 121 ms, sys: 11.3 ms, total: 132 ms
Wall time: 52.2 ms


## Pregunta 5. Outliers por percentiles 1 y 99

In [11]:
%%time
percentiles = clean.select(
    balance_p01=pl.col(BALANCE).quantile(0.01, interpolation='linear'),
    balance_p99=pl.col(BALANCE).quantile(0.99, interpolation='linear'),
    amount_p01=pl.col(AMOUNT).quantile(0.01, interpolation='linear'),
    amount_p99=pl.col(AMOUNT).quantile(0.99, interpolation='linear'),
).row(0, named=True)

outliers = clean.select(
    balance_outliers=(
        (pl.col(BALANCE) < percentiles['balance_p01'])
        | (pl.col(BALANCE) > percentiles['balance_p99'])
    ).sum(),
    amount_outliers=(
        (pl.col(AMOUNT) < percentiles['amount_p01'])
        | (pl.col(AMOUNT) > percentiles['amount_p99'])
    ).sum(),
).row(0, named=True)

q05 = pl.DataFrame({
    'variable': [BALANCE, AMOUNT],
    'p01': [percentiles['balance_p01'], percentiles['amount_p01']],
    'p99': [percentiles['balance_p99'], percentiles['amount_p99']],
    'outlier_rows': [outliers['balance_outliers'], outliers['amount_outliers']],
})
q05.write_csv(OUTPUT_DIR / 'q05_outliers.csv')
display(q05)

variable,p01,p99,outlier_rows
str,f64,f64,i64
"""CustAccountBalance""",3.23,1.5869e6,20913
"""TransactionAmount (INR)""",8.0,20000.0,20407


CPU times: user 208 ms, sys: 10.1 ms, total: 218 ms
Wall time: 71.1 ms


## Pregunta 6. Balance y monto promedio por género y edad

In [12]:
%%time
q06 = clean.filter(
    pl.col('CustGender').is_not_null() & pl.col('Age').is_not_null()
).group_by(['CustGender', 'AgeRange']).agg(
    transactions=pl.len(),
    avg_balance=pl.col(BALANCE).mean().round(2),
    avg_amount=pl.col(AMOUNT).mean().round(2),
).sort(['CustGender', 'AgeRange'])
q06.write_csv(OUTPUT_DIR / 'q06_genero_edad.csv')
display(q06)

CustGender,AgeRange,transactions,avg_balance,avg_amount
str,str,u32,f64,f64
"""F""","""18-25""",94727,37858.98,1007.18
"""F""","""26-35""",125778,84128.28,1620.86
"""F""","""36-45""",34248,200026.15,2323.98
"""F""","""46-60""",14126,267580.73,3171.41
"""F""","""61-100""",4210,695495.59,3080.99
"""M""","""18-25""",200390,33768.21,803.19
"""M""","""26-35""",357212,84804.58,1275.34
"""M""","""36-45""",108337,190350.65,2155.85
"""M""","""46-60""",36730,348567.61,2973.25


CPU times: user 265 ms, sys: 121 ms, total: 386 ms
Wall time: 141 ms


## Pregunta 7. Top 20 ciudades

In [13]:
%%time
q07 = clean.filter(
    pl.col('LocationNormalized').is_not_null()
).group_by('LocationNormalized').agg(
    transactions=pl.len(),
    total_amount=pl.col(AMOUNT).sum().round(2),
).sort(
    ['total_amount', 'LocationNormalized'], descending=[True, False]
).head(20)
q07.write_csv(OUTPUT_DIR / 'q07_top_ciudades.csv')
display(q07)

LocationNormalized,transactions,total_amount
str,u32,f64
"""MUMBAI""",103596,1.7969e8
"""NEW DELHI""",84928,1.6071e8
"""BANGALORE""",81555,1.1842e8
"""GURGAON""",73818,1.1209e8
"""DELHI""",71019,1.0622e8
…,…,…
"""CHANDIGARH""",9526,1.4919e7
"""JAIPUR""",9921,1.4067e7
"""LUCKNOW""",7763,1.1903e7


CPU times: user 248 ms, sys: 97.9 ms, total: 346 ms
Wall time: 116 ms


## Pregunta 8. Cliente con mayor gasto por ciudad

In [14]:
%%time
spending = clean.filter(
    pl.col('LocationNormalized').is_not_null() & pl.col('CustomerID').is_not_null()
).group_by(['LocationNormalized', 'CustomerID']).agg(
    total_spent=pl.col(AMOUNT).sum(),
    transactions=pl.len(),
)

q08 = spending.sort(
    ['LocationNormalized', 'total_spent', 'CustomerID'],
    descending=[False, True, False],
).with_columns(
    city_rank=pl.int_range(1, pl.len() + 1).over('LocationNormalized')
).filter(
    pl.col('city_rank') == 1
).with_columns(
    total_spent=pl.col('total_spent').round(2)
).select(
    ['LocationNormalized', 'CustomerID', 'total_spent', 'transactions', 'city_rank']
).sort(['total_spent', 'LocationNormalized'], descending=[True, False])

q08.write_csv(OUTPUT_DIR / 'q08_top_cliente_ciudad.csv')
print('Ciudades:', q08.height)
display(q08.head(20))

Ciudades: 9353


LocationNormalized,CustomerID,total_spent,transactions,city_rank
str,str,f64,u32,i64
"""GURGAON""","""C7319271""",1.5600e6,1,1
"""PUNE""","""C6677159""",1.3800e6,1,1
"""NEW DELHI""","""C4141768""",991132.22,1,1
"""MUMBAI""","""C8217728""",724122.0,1,1
"""KOLKATA""","""C1830891""",720001.16,1,1
…,…,…,…,…
"""JAIPUR DURGAPURA""","""C5727148""",378415.46,1,1
"""RAIPUR""","""C6741078""",378006.07,1,1
"""ANAND""","""C5140623""",326510.0,1,1


CPU times: user 2.32 s, sys: 179 ms, total: 2.49 s
Wall time: 818 ms


## Pregunta 9. Serie temporal diaria

In [15]:
%%time
q09 = clean.filter(
    pl.col('TransactionDateParsed').is_not_null()
).group_by('TransactionDateParsed').agg(
    transactions=pl.len(),
    total_amount=pl.col(AMOUNT).sum().round(2),
).sort('TransactionDateParsed')
q09.write_csv(OUTPUT_DIR / 'q09_serie_diaria.csv')
display(q09)

TransactionDateParsed,transactions,total_amount
date,u32,f64
2016-08-01,20438,2.9802e7
2016-08-02,20948,3.0468e7
2016-08-03,20615,3.1149e7
2016-08-04,20682,3.5723e7
2016-08-05,21112,3.4834e7
…,…,…
2016-09-26,12460,2.0782e7
2016-09-27,7447,1.0902e7
2016-09-30,1951,3316668.5


CPU times: user 126 ms, sys: 38.4 ms, total: 164 ms
Wall time: 53.3 ms


## Pregunta 10. Ratio gasto/balance y top 1%

In [16]:
%%time
valid = clean.filter(
    pl.col(BALANCE).is_not_null() & (pl.col(BALANCE) > 0) & pl.col(AMOUNT).is_not_null()
).with_columns(
    spend_balance_ratio=pl.col(AMOUNT) / pl.col(BALANCE)
)

top_count = math.ceil(valid.height * 0.01)
q10 = valid.sort(
    ['spend_balance_ratio', 'TransactionID'], descending=[True, False]
).head(top_count).select([
    'TransactionID', 'CustomerID', 'LocationNormalized',
    BALANCE, AMOUNT, 'spend_balance_ratio',
])
q10.write_csv(OUTPUT_DIR / 'q10_ratio_top1.csv')
print('Filas válidas:', valid.height, '| Filas del top 1%:', q10.height)
display(q10.head(20))

Filas válidas: 1043487 | Filas del top 1%: 10435


TransactionID,CustomerID,LocationNormalized,CustAccountBalance,TransactionAmount (INR),spend_balance_ratio
str,str,str,f64,f64,f64
"""T742111""","""C7323566""","""CHANDIGARH""",0.01,42398.0,4.2398e6
"""T836117""","""C5719489""","""KOLAR""",0.01,25500.0,2.55e6
"""T253453""","""C3523520""","""CHANDIGARH""",0.01,20000.0,2e6
"""T652735""","""C6038911""","""TANK HYDERABAD""",0.01,17820.0,1.782e6
"""T421850""","""C2816416""","""MUMBAI""",0.01,15715.0,1.5715e6
…,…,…,…,…,…
"""T872230""","""C5930460""","""NEW DELHI""",0.01,5550.0,555000.0
"""T285555""","""C4099744""","""KAITHAL""",0.01,5280.0,528000.0
"""T60477""","""C1714915""","""NAVI MUMBAI""",0.01,4978.5,497850.0


CPU times: user 809 ms, sys: 214 ms, total: 1.02 s
Wall time: 340 ms
